# Advanced Lab · Flow Matching / Action Chunk

Optional lab. Complete Chapters 06–07 first. Here an action model predicts a short trajectory chunk, and the planner decides how many steps to execute before re-planning. The point is to connect a generative mechanism to action-space, horizon, latency and closed-loop drift—not to move it into the prerequisite path.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import matplotlib.pyplot as plt

prediction = load_numpy_artifact("06_prediction.npz")
target = prediction["truth_future"]
rng = np.random.default_rng(3)
noise = rng.normal(0, 1.0, target.shape)
times = np.linspace(0.0, 1.0, len(target))[:, None]
samples = noise * (1.0 - times) + target * times

In [ ]:
def integrate_flow(initial, target, steps=8):
    state = initial.copy()
    for t in np.linspace(0.0, 1.0, steps):
        velocity = target - state
        state = state + velocity / steps
    return state

initial = np.zeros_like(target)
for steps in [2, 4, 8, 16]:
    generated = integrate_flow(initial, target, steps=steps)
    print("steps", steps, "terminal error", np.linalg.norm(generated[-1] - target[-1]).round(3))
plt.plot(target[:, 0], target[:, 1], linewidth=3, label="target action chunk")
plt.plot(samples[:, 0], samples[:, 1], "--", label="flow interpolation")
plt.legend(); plt.axis("equal"); plt.title("Action chunk as a trajectory object"); plt.show()

练习：改变 action chunk 长度、replan frequency 和 initial noise；报告 terminal error、closed-loop minimum gap 和 p95 latency。解释为什么更长 chunk 可能减少 compute，却增加 model-mismatch exposure。